# Benchmark Dynamical Localization in Floquet System

Dynamical localization is a characteristic quantum effect in periodically driven chaotic systems. In classical maps, namely discrete-time dynamical systems of the form $x_t \rightarrow x_{t+1}$, an initially localized distribution typically spreads over time in a diffusive manner. In their quantized counterparts, known as quantum maps, interference can suppress this spreading and keep the wavepacket concentrated around its initial momentum. Because this genuinely quantum effect relies on the coherent accumulation of phase over several time steps, while still admitting a relatively simple circuit implementation, it has been proposed as a clear benchmarking task for quantum hardware [[1]](#Pizzamiglio_etal). In particular, here we consider the quantum sawtooth map.

***

### The model

We benchmark dynamical localization using the quantum sawtooth map, whose unitary evolution in momentum space after $T$ time steps is given by
$$
|\psi_T\rangle
=
\left(
\exp\left[-i\frac{2\pi}{N}\frac{\hat{p}^2}{2}\right]
\exp\left[iK\frac{2\pi}{N}\frac{\hat{q}^2}{2}\right]
\right)^T
|\psi_0\rangle.
$$

The model initializes the momentum register in a localized basis state,
$$
|\psi(0)\rangle = |-2\rangle,
$$
and evolves it for different final times $T$. To probe whether the state remains localized, we evaluate the output distribution after two and three kicks, namely at $T=2$ and $T=3$, and examine how much probability remains on the initial momentum value $|-2\rangle$.

### The scoring function

The scoring function is designed to quantify the persistence of the localization peak over time. Let
$$
P_T(-2)
$$
denote the measured probability of observing the momentum value $|-2\rangle$ after $T$ kicks. In a localized regime, this peak should remain significantly larger than the uniform background at both times. In contrast, delocalization, noise, or control errors tend to reduce the peak as probability spreads across the momentum register.

For an $n$-qubit register, the uniform baseline is $1/N$, where
$$
N = 2^n.
$$
We therefore define the normalized peak strength at time $T$ by
$$
\tilde{P}_T
=
\frac{P_T(-2)-1/N}{1-1/N},
$$
with values clipped to the interval $[0,1]$.

The final localization score is then defined as the geometric mean of the normalized peak strengths at the two sampled times,
$$
S = \sqrt{\tilde{P}_2 \tilde{P}_3}.
$$

This score rewards both the presence of a strong localization peak and its persistence over successive kicks. A value $S=0$ corresponds to a peak no better than the uniform background, while larger values indicate stronger and more stable localization around the initial momentum.

Since the system is finite, localization is not expected to produce a perfectly stationary or monotonic peak. Instead, the probability distribution may remain concentrated near the initial momentum while still exhibiting small oscillations or partial revivals. Accordingly, the score evaluates the strength of the localization peak at two successive times, rather than enforcing monotonic decay or exact time-independence. Because this is an absolute localization score for a finite system, even an ideal noiseless simulation need not attain the maximal value $S=1$.

***
***

## Defining a `BenchmarkExample` for the Quantum Sawtooth Map

In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *

In [ ]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project
# reset_benchmark_project()
# ============================================================

In [2]:
from classiq.qmod.symbolic import pi


@qfunc
def single_step(kick_mag: CReal, p: QNum):
    within_apply(lambda: qft(p), lambda: phase((kick_mag * pi / (2**p.size)) * p**2))
    phase(-(pi / 2**p.size) * p**2)

In [3]:
LOCALIZATION_DESCRIPTION = Path("../descriptions/dynamical_localization.tex").read_text(
    encoding="utf-8"
)

In [4]:
class QSMExample(BenchmarkExample):
    def __init__(self):
        super().__init__(
            name="localization",
            num_qubits=3,
            report_family_title="Dynamical Localization",
            report_family_description=LOCALIZATION_DESCRIPTION,
        )

    def create_main(self) -> callable:
        @qfunc
        def main(num_kicks: CInt, p: Output[QNum[self.num_qubits, SIGNED, 0]]):
            allocate(p)
            p ^= -2
            power(num_kicks, lambda: single_step(0.1, p))

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_batch_sample([{"num_kicks": 2}, {"num_kicks": 3}])
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()
        df_t_2 = result[0].value.details[0].dataframe
        df_t_3 = result[0].value.details[0].dataframe

        def get_peak_prob(df):
            match = df.loc[df["p"] == -2, "probability"]
            return 0.0 if match.empty else float(match.iloc[0])

        def normalize_peak(p):
            N = 2**self.num_qubits
            return max(0.0, min(1.0, (p - 1 / N) / (1 - 1 / N)))

        p2 = get_peak_prob(df_t_2)
        p3 = get_peak_prob(df_t_3)

        s2 = normalize_peak(p2)
        s3 = normalize_peak(p3)

        score = (s2 * s3) ** 0.5

        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": score,
            "execution_time": exec_minutes,
        }

***
***
## Benchmarking Dynamical Localization on 3-qubits

Define a specific example on 3 qubits (the size is fixed for this example)

In [5]:
example = QSMExample()
example.show()

Quantum program link: https://platform.classiq.io/circuit/3BS8NcEaTLQD23vLIOOaXKcWkyN


### Set a `ResultCollector` with a file name fixed for the current example

In [6]:
FILENAME = example.default_results_filename

In [7]:
collector = ResultCollector(FILENAME, build_each_time=True)

### Choose on which backend to run 

We can print the list of backends:

In [8]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [9]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [10]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [11]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [12]:
print(
    "=" * 10
    + f"  {example.name}-{example.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  localization-3 (2026-03-25 22:37:53.503317)   ==========
2026-03-25 22:37:57.146562: Submit localization-3 for Classiq - simulator_statevector
2026-03-25 22:37:58.397532: Completed localization-3 for Classiq - simulator_statevector --> Score {'score': 0.9147397211610553, 'execution_time': 0.014012783333333334}
** Report updated: localization-3 for Classiq - simulator_statevector


In [13]:
await collector.print_status()

========== (2026-03-25 22:37:59.428299)   ==========
localization-3 | Classiq - simulator | COMPLETED | score=0.9154 | time=0.02 min
localization-3 | Classiq - simulator_statevector | COMPLETED | score=0.9147 | time=0.01 min


## Refrences

<a id='Pizzamiglio_etal'>[1]</a>: [Andrea Pizzamiglio, Su Yeon Chang, Maria Bondani, Simone Montangero, Dario Gerace, Giuliano Benenti. "Dynamical localization simulated on actual quantum hardware." Entropy 23, 654 (2021).](https://arxiv.org/abs/2105.10813)